# Silver Layer — E-commerce


In [ ]:
CREATE SCHEMA IF NOT EXISTS azure_databricks_dev.silver;


In [ ]:
CREATE OR REPLACE TABLE azure_databricks_dev.silver.users
USING DELTA
AS
SELECT
  TRIM(user_id) AS user_id,
  TRIM(name) AS name,
  LOWER(TRIM(email)) AS email,
  INITCAP(TRIM(gender)) AS gender,
  TRIM(city) AS city,
  TO_DATE(signup_date) AS signup_date
FROM azure_databricks_dev.bronze.users_raw
WHERE user_id IS NOT NULL
  AND TRIM(user_id) <> ''
  AND email IS NOT NULL
  AND TRIM(email) <> '';


In [ ]:
CREATE OR REPLACE TABLE azure_databricks_dev.silver.products
USING DELTA
AS
SELECT
  TRIM(product_id) AS product_id,
  TRIM(product_name) AS product_name,
  INITCAP(TRIM(category)) AS category,
  TRIM(brand) AS brand,
  CAST(price AS DECIMAL(18,2)) AS price,
  CAST(rating AS DECIMAL(3,2)) AS rating
FROM azure_databricks_dev.bronze.products_raw
WHERE product_id IS NOT NULL
  AND TRIM(product_id) <> ''
  AND price IS NOT NULL
  AND price >= 0
  AND rating BETWEEN 0 AND 5;


In [ ]:
CREATE OR REPLACE TABLE azure_databricks_dev.silver.orders
USING DELTA
AS
SELECT
  TRIM(order_id) AS order_id,
  TRIM(user_id) AS user_id,
  TO_TIMESTAMP(order_date) AS order_date,
  LOWER(TRIM(order_status)) AS order_status,
  CAST(total_amount AS DECIMAL(18,2)) AS total_amount
FROM azure_databricks_dev.bronze.orders_raw
WHERE order_id IS NOT NULL
  AND TRIM(order_id) <> ''
  AND user_id IS NOT NULL
  AND TRIM(user_id) <> ''
  AND total_amount IS NOT NULL
  AND total_amount >= 0;


In [ ]:
CREATE OR REPLACE TABLE azure_databricks_dev.silver.order_items
USING DELTA
AS
SELECT
  TRIM(order_item_id) AS order_item_id,
  TRIM(order_id) AS order_id,
  TRIM(product_id) AS product_id,
  TRIM(user_id) AS user_id,
  CAST(quantity AS INT) AS quantity,
  CAST(item_price AS DECIMAL(18,2)) AS item_price,
  CAST(item_total AS DECIMAL(18,2)) AS item_total
FROM azure_databricks_dev.bronze.order_items_raw
WHERE order_item_id IS NOT NULL
  AND order_id IS NOT NULL
  AND product_id IS NOT NULL
  AND quantity > 0
  AND item_price >= 0
  AND item_total >= 0;


In [ ]:
CREATE OR REPLACE TABLE azure_databricks_dev.silver.events
USING DELTA
AS
SELECT
  TRIM(event_id) AS event_id,
  TRIM(user_id) AS user_id,
  TRIM(product_id) AS product_id,
  LOWER(TRIM(event_type)) AS event_type,
  TO_TIMESTAMP(event_timestamp) AS event_timestamp
FROM azure_databricks_dev.bronze.events_raw
WHERE event_id IS NOT NULL
  AND user_id IS NOT NULL
  AND product_id IS NOT NULL
  AND event_timestamp IS NOT NULL;


In [ ]:
CREATE OR REPLACE TABLE azure_databricks_dev.silver.reviews
USING DELTA
AS
SELECT
  TRIM(review_id) AS review_id,
  TRIM(order_id) AS order_id,
  TRIM(product_id) AS product_id,
  TRIM(user_id) AS user_id,
  CAST(rating AS INT) AS rating,
  TRIM(review_text) AS review_text,
  TO_TIMESTAMP(review_date) AS review_date
FROM azure_databricks_dev.bronze.reviews_raw
WHERE review_id IS NOT NULL
  AND product_id IS NOT NULL
  AND user_id IS NOT NULL
  AND rating BETWEEN 1 AND 5;


In [ ]:
-- Silver validation: duplicate business keys
SELECT 'users' AS table_name, COUNT(*) - COUNT(DISTINCT user_id) AS duplicate_keys
FROM azure_databricks_dev.silver.users
UNION ALL
SELECT 'products', COUNT(*) - COUNT(DISTINCT product_id)
FROM azure_databricks_dev.silver.products
UNION ALL
SELECT 'orders', COUNT(*) - COUNT(DISTINCT order_id)
FROM azure_databricks_dev.silver.orders
UNION ALL
SELECT 'order_items', COUNT(*) - COUNT(DISTINCT order_item_id)
FROM azure_databricks_dev.silver.order_items
UNION ALL
SELECT 'events', COUNT(*) - COUNT(DISTINCT event_id)
FROM azure_databricks_dev.silver.events
UNION ALL
SELECT 'reviews', COUNT(*) - COUNT(DISTINCT review_id)
FROM azure_databricks_dev.silver.reviews;
